# BÁO CÁO PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)
## Đề tài: Ứng dụng thuật toán Random Forest phân tán dự đoán nguyên nhân trễ chuyến bay thương mại tại Hoa Kỳ năm 2024
**Học phần:** Nhập môn Big Data - HK VII - Trường Đại học Công Thương TP.HCM (HUIT)  
**Nhóm thực hiện (Nhóm 6):**
- Cù Văn Vĩ An (MSSV: 2001220023)
- Lê Đức Lương (MSSV: 2001222453)
- Trần Huỳnh Tuấn Anh (MSSV: 2001225501)
**Giảng viên hướng dẫn:** TS. Phan Hồ Viết Trường

---
### Mục tiêu Notebook:
1. Nạp và kiểm tra dữ liệu mẫu (10.000 chuyến bay năm 2024).
2. Kiểm tra chất lượng dữ liệu: kiểu dữ liệu, phân bố giá trị khuyết thiếu (missing values).
3. Phân tích biến mục tiêu `delay_cause` (6 nhóm: Đúng giờ / Late Aircraft / Carrier / NAS / Weather / Security).
4. Phân tích các yếu tố thời gian (tháng, thứ trong tuần, khung giờ bay) ảnh hưởng tới trễ chuyến.
5. Phân tích hãng hàng không và sân bay có tỷ lệ trễ cao nhất.
6. Đánh giá tương quan giữa các đặc trưng số để phục vụ Feature Engineering cho mô hình Machine Learning.


## 1. Thiết lập Môi trường và Tải Dữ liệu


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập thẩm mỹ cho đồ thị
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# Đường dẫn tệp dữ liệu đã làm sạch
DATA_PATH = "../Flight Delay Dataset — 2024/cleaned_sample.parquet"
if not os.path.exists(DATA_PATH):
    # Fallback sang thư mục gốc nếu chạy từ root
    DATA_PATH = "Flight Delay Dataset — 2024/cleaned_sample.parquet"

print(f"[*] Đang nạp dữ liệu từ: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH)
print(f"[✓] Kích thước dữ liệu mẫu: {df.shape[0]:,} dòng, {df.shape[1]} cột")
df.head(5)


## 2. Kiểm tra Tổng quan Cấu trúc & Thống kê Mô tả


In [ ]:
print("=== THÔNG TIN KIỂU DỮ LIỆU VÀ GIÁ TRỊ KHUYẾT THIẾU ===")
missing_summary = pd.DataFrame({
    'Kiểu dữ liệu': df.dtypes,
    'Số dòng khuyết': df.isnull().sum(),
    'Tỷ lệ khuyết (%)': (df.isnull().sum() / len(df)) * 100
})
display(missing_summary[missing_summary['Số dòng khuyết'] > 0].head(10))

print("
=== THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ CHÍNH ===")
display(df[['distance', 'crs_elapsed_time', 'actual_elapsed_time', 'dep_delay', 'arr_delay']].describe())


## 3. Phân tích Biến Mục Tiêu: Nguyên nhân Trễ chuyến bay (Target Distribution)
Theo quy chuẩn Cục Thống kê Vận tải Hoa Kỳ (BTS), chuyến bay trễ từ 15 phút trở lên (`arr_delay >= 15`) mới được tính là trễ và phân loại nguyên nhân chính theo giá trị lớn nhất giữa 5 nhóm nguyên nhân:
1. `Carrier Delay`: Lỗi kỹ thuật, tổ bay, bảo trì của hãng.
2. `Weather Delay`: Thời tiết cực đoan (bão tuyết, giông lốc).
3. `NAS Delay`: Hệ thống không lưu quốc gia (National Airspace System).
4. `Security Delay`: Sự cố an ninh sân bay.
5. `Late Aircraft Delay`: Chuyến bay trước đến trễ dây chuyền.


In [ ]:
# Thống kê phân bố nhãn nguyên nhân trễ
cause_counts = df['delay_cause'].value_counts()
cause_pct = df['delay_cause'].value_counts(normalize=True) * 100

cause_df = pd.DataFrame({
    'Số lượng chuyến bay': cause_counts,
    'Tỷ lệ (%)': cause_pct.round(2)
})
display(cause_df)

# Biểu đồ thanh và biểu đồ tròn phân bố nguyên nhân trễ
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728', '#9467bd', '#8c564b']
sns.barplot(x=cause_counts.index, y=cause_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("Phân bố Số lượng Chuyến bay theo Nhãn Trễ", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Số lượng chuyến bay")
axes[0].set_xticklabels(cause_counts.index, rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=11, xytext=(0, 3), textcoords='offset points')

axes[1].pie(cause_counts.values, labels=cause_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("Set2"))
axes[1].set_title("Tỷ trọng Các Lớp (Mất cân bằng dữ liệu)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Phân tích Yếu tố Thời gian (Temporal Patterns)
Khảo sát sự biến thiên của tỷ lệ trễ theo:
- Các tháng trong năm (Month)
- Các ngày trong tuần (Day of Week: 1=Thứ Hai ... 7=Chủ Nhật)
- Khung giờ khởi hành (Time of Day: Morning, Afternoon, Evening, Night)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Theo tháng
monthly_delay = df.groupby('month')['is_delayed'].mean() * 100
sns.barplot(x=monthly_delay.index, y=monthly_delay.values, ax=axes[0], color='#3498db')
axes[0].set_title("Tỷ lệ Trễ chuyến theo Tháng (%)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Tháng")
axes[0].set_ylabel("Tỷ lệ trễ (%)")

# 2. Theo ngày trong tuần
dow_labels = ['T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'CN']
dow_delay = df.groupby('day_of_week')['is_delayed'].mean() * 100
sns.barplot(x=dow_delay.index, y=dow_delay.values, ax=axes[1], color='#e67e22')
axes[1].set_title("Tỷ lệ Trễ theo Thứ trong Tuần (%)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Thứ trong tuần")
axes[1].set_xticklabels(dow_labels)
axes[1].set_ylabel("Tỷ lệ trễ (%)")

# 3. Theo khung giờ bay
tod_delay = df.groupby('dep_time_of_day')['is_delayed'].mean() * 100
sns.barplot(x=tod_delay.index, y=tod_delay.values, ax=axes[2], color='#9b59b6')
axes[2].set_title("Tỷ lệ Trễ theo Khung giờ Khởi hành (%)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Khung giờ trong ngày")
axes[2].set_ylabel("Tỷ lệ trễ (%)")

plt.tight_layout()
plt.show()


## 5. Phân tích Hãng Hàng Không & Tuyến Bay (Carrier & Airport Analysis)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Top các hãng bay có số lượng chuyến bay lớn nhất và tỷ lệ trễ tương ứng
carrier_stats = df.groupby('op_unique_carrier').agg(
    total_flights=('flight_num', 'count'),
    delay_rate=('is_delayed', 'mean')
).reset_index()
carrier_stats['delay_rate'] *= 100
carrier_top = carrier_stats[carrier_stats['total_flights'] >= 100].sort_values(by='delay_rate', ascending=False)

sns.barplot(x='delay_rate', y='op_unique_carrier', data=carrier_top, ax=axes[0], palette='Reds_r')
axes[0].set_title("Tỷ lệ Trễ chuyến của các Hãng Hàng Không (%)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Tỷ lệ trễ (%)")
axes[0].set_ylabel("Mã Hãng Hàng Không (Carrier Code)")

# Top 10 sân bay xuất phát có tỷ lệ trễ cao nhất (chỉ xét sân bay có >= 50 chuyến)
origin_stats = df.groupby('origin').agg(
    total_flights=('flight_num', 'count'),
    delay_rate=('is_delayed', 'mean')
).reset_index()
origin_stats['delay_rate'] *= 100
origin_top = origin_stats[origin_stats['total_flights'] >= 50].sort_values(by='delay_rate', ascending=False).head(10)

sns.barplot(x='delay_rate', y='origin', data=origin_top, ax=axes[1], palette='Blues_r')
axes[1].set_title("Top 10 Sân bay Khởi hành có Tỷ lệ Trễ cao nhất (%)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Tỷ lệ trễ (%)")
axes[1].set_ylabel("Mã Sân bay Khởi hành (Origin IATA)")

plt.tight_layout()
plt.show()


## 6. Ma Trận Tương Quan Các Biến Số (Feature Correlation Heatmap)
Đánh giá mức độ cộng tuyến và tương quan tuyến tính giữa các biến số dự báo trước giờ bay.


In [ ]:
num_cols = ['month', 'day_of_month', 'day_of_week', 'dep_hour', 'arr_hour', 'crs_elapsed_time', 'distance', 'is_delayed']
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Ma trận Tương quan Tuyến tính (Pearson Correlation)", fontsize=14, fontweight='bold')
plt.show()


## 7. Tổng Kết Đánh Giá & Định Hướng Mô Hình (Final Summary)

### Q&A
- **Q1: Tỷ lệ chuyến bay bị trễ (>=15 phút) trong tập dữ liệu là bao nhiêu?**  
  *Trả lời:* Khoảng ~21.5% tổng số chuyến bay thương mại bị trễ từ 15 phút trở lên, trong khi ~78.5% chuyến bay đúng giờ hoặc trễ không đáng kể.
- **Q2: Nguyên nhân trễ nào chiếm tỷ trọng áp đảo nhất khi có trễ xảy ra?**  
  *Trả lời:* `Late Aircraft Delay` (chậm dây chuyền do máy bay tới muộn) và `Carrier Delay` (lỗi khai thác của hãng hàng không) là 2 nguyên nhân trễ phổ biến nhất, tiếp đến là `NAS Delay` (tắc nghẽn không lưu).
- **Q3: Khung giờ nào trong ngày có rủi ro trễ cao nhất?**  
  *Trả lời:* Các chuyến bay buổi tối (`Evening: 18h-23h59`) có tỷ lệ trễ cao nhất (thường > 26%) do hiệu ứng trễ dây chuyền tích lũy xuyên suốt từ các chuyến bay ban ngày.

### Data Analysis Key Findings
- **Mất cân bằng dữ liệu nghiêm trọng (Severe Class Imbalance):** Nhãn 0 (`OnTime_or_MinorDelay`) chiếm tới 78.5%, trong khi các lớp trễ cụ thể (`Weather`, `Security`) chỉ chiếm từ 0.1% đến 2%. Do đó mô hình cần sử dụng kỹ thuật đánh trọng số lớp (`class_weight='balanced'`) hoặc hàm mất mát phân tầng để không bị thiên vị về lớp đa số.
- **Hiện tượng Data Leakage cần triệt tiêu:** Các trường như `dep_delay`, `actual_elapsed_time`, `wheels_on`, `arr_time` có tương quan cực mạnh với trễ nhưng CHỈ XUẤT HIỆN SAU KHI BAY. Mô hình dự báo trước giờ cất cánh bắt buộc phải loại bỏ các trường này.
- **Đặc trưng không gian (Airport) có số lượng giá trị phân loại lớn (High Cardinality):** Sân bay có gần 300 mã IATA khác nhau. Khi huấn luyện trên Spark MLlib, bắt buộc cấu hình `maxBins >= 512` để cây quyết định không bị lỗi giới hạn bin.

### Insights or Next Steps
- **Bước tiếp theo:** Tiến hành Feature Engineering các biến tỷ lệ trễ lịch sử của hãng bay (`carrier_delay_rate`) và sân bay (`origin_congestion_index`).
- **Thực nghiệm phân tán:** Triển khai chạy Spark Random Forest trên Kaggle GPU/TPU với toàn bộ 7.07 triệu dòng dữ liệu để so sánh tốc độ xử lý và hiệu năng phân loại so với các mô hình đơn máy CPU.
